<a href="https://colab.research.google.com/github/abiya-pardeshi/traffic-surveillance-dashboard/blob/main/Traffic_Surveillance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

#JUNCTIONS & CAMERAS
junctions = pd.DataFrame({
    "junction_id": [f"J{i:02d}" for i in range(1, 11)],
    "name": ["Market Circle", "Bus Stand Junction", "River Bridge", "Seaface Circle",
             "Airport Road", "College Junction", "Hospital Circle", "Station Junction",
             "Flyover North", "Causeway East"],
    "zone": ["Central", "Central", "North", "South", "South",
             "Central", "Central", "East", "North", "East"],
    "lat": [15.4989, 15.4909, 15.5060, 15.4830, 15.4540,
            15.4850, 15.4880, 15.5020, 15.5270, 15.5040],
    "lon": [73.8278, 73.8300, 73.8350, 73.8085, 73.8100,
            73.8180, 73.8260, 73.8340, 73.8240, 73.8650],
    "camera_id": [f"CAM{i:02d}" for i in range(1, 11)],
    "camera_status": ["Online"] * 8 + ["Offline", "Maintenance"],
    "uptime_pct": np.round(rng.uniform(92, 99.5, 10), 1),
    "free_flow_speed": 50,
})
# busy junctions produce more traffic and more violations (hotspots)
weights = dict(zip(junctions.junction_id,
                   [0.18, 0.16, 0.14, 0.08, 0.05, 0.10, 0.09, 0.08, 0.07, 0.05]))

# HOURLY TRAFFIC FLOW
hours = pd.date_range("2025-11-01", "2026-10-28 23:00", freq="h")
h, dow, m = hours.hour.values, hours.dayofweek.values, hours.month.values

hour_f = (0.15 + 0.85*np.exp(-((h-9)**2)/4)         # morning rush ~9am
          + 1.0*np.exp(-((h-18.5)**2)/5)            # evening rush ~6:30pm
          + 0.35*np.exp(-((h-13)**2)/10))           # small lunch bump
week_f = np.where(dow >= 5, 0.8, 1.0)               # weekends are quieter
monsoon = np.isin(m, [6, 7, 8, 9])
season_f = np.where(np.isin(m, [11, 12, 1]), 1.15,  # tourist season
                    np.where(monsoon, 0.9, 1.0))

frames = []
for _, j in junctions.iterrows():
    size = 500 + weights[j.junction_id] * 5000
    count = rng.poisson(size * hour_f * week_f * season_f)
    load = count / (size * 1.5)
    speed = 50 * (1 - 0.55 * load**1.5) - monsoon * 4 + rng.normal(0, 2, len(hours))
    speed = np.clip(speed, 8, 50).round(1)
    weather = np.where(monsoon,
                       rng.choice(["Rain", "Cloudy", "Clear"], len(hours), p=[.5, .3, .2]),
                       rng.choice(["Rain", "Cloudy", "Clear"], len(hours), p=[.03, .27, .7]))
    frames.append(pd.DataFrame({
        "timestamp": hours, "junction_id": j.junction_id,
        "vehicle_count": count, "avg_speed_kmph": speed,
        "congestion_index": (1 - speed / 50).round(3),
        "weather": weather,
    }))
traffic = pd.concat(frames, ignore_index=True)

#VIOLATIONS
N = 32000
t = ((traffic.timestamp - traffic.timestamp.min()) /
     (traffic.timestamp.max() - traffic.timestamp.min())).values
w = traffic.vehicle_count.values * traffic.junction_id.map(weights).values * (1 + 0.25*t)
idx = rng.choice(len(traffic), size=N, p=w / w.sum())
v = traffic.iloc[idx][["timestamp", "junction_id"]].reset_index(drop=True)
v["timestamp"] += pd.to_timedelta(rng.integers(0, 3600, N), unit="s")

types = ["No Helmet", "Over Speeding", "Signal Jump", "Wrong Parking",
         "No Seatbelt", "Triple Riding", "Mobile Use"]
fines = {"No Helmet": 500, "Over Speeding": 1000, "Signal Jump": 1000,
         "Wrong Parking": 500, "No Seatbelt": 1000, "Triple Riding": 1000, "Mobile Use": 1000}
v["violation_type"] = rng.choice(types, N, p=[.30, .18, .15, .18, .08, .06, .05])
v["fine_amount"] = v.violation_type.map(fines)

vt = rng.choice(["Two-Wheeler", "Car", "Auto", "Truck", "Bus"], N, p=[.55, .28, .08, .05, .04])
vt = np.where(v.violation_type.isin(["No Helmet", "Triple Riding"]), "Two-Wheeler", vt)
vt = np.where(v.violation_type == "No Seatbelt", "Car", vt)
v["vehicle_type"] = vt

speeding = v.violation_type == "Over Speeding"
v["speed_limit"] = np.where(speeding, 50, np.nan)
v["speed_recorded"] = np.where(speeding, 50 + rng.integers(10, 41, N), np.nan)

# repeat offenders: some vehicles appear many times
pool = [f"GA{rng.integers(1,13):02d}{chr(65+rng.integers(0,26))}{chr(65+rng.integers(0,26))}{rng.integers(1000,9999)}"
        for _ in range(9000)]
p = rng.pareto(2.5, 9000) + 1
v["vehicle_number"] = rng.choice(pool, N, p=p / p.sum())
v["camera_id"] = v.junction_id.map(dict(zip(junctions.junction_id, junctions.camera_id)))
v.insert(0, "violation_id", [f"V{i:06d}" for i in range(1, N + 1)])

# CHALLANS
today = pd.Timestamp("2026-10-28")
issue = v.timestamp.dt.normalize() + pd.to_timedelta(rng.integers(1, 4, N), unit="D")
due = issue + pd.Timedelta(days=30)
will_pay = rng.random(N) < 0.78
late = rng.random(N) < 0.20
days = np.where(late, rng.integers(31, 91, N), rng.integers(1, 31, N))
pay_date = issue + pd.to_timedelta(days, unit="D")
paid = will_pay & (pay_date <= today)
contested = (~paid) & (rng.random(N) < 0.06)
status = np.select([paid, contested, due < today], ["Paid", "Contested", "Overdue"], "Pending")

challans = pd.DataFrame({
    "challan_id": [f"C{i:06d}" for i in range(1, N + 1)],
    "violation_id": v.violation_id,
    "issue_date": issue.dt.date, "due_date": due.dt.date,
    "status": status,
    "payment_date": pd.Series(pay_date.where(paid).dt.date),
    "amount_paid": np.where(paid, v.fine_amount, 0),
})

# AVE
enriched = (v.merge(challans, on="violation_id")
             .merge(junctions[["junction_id", "name", "zone", "lat", "lon"]], on="junction_id"))
traffic_full = traffic.merge(junctions[["junction_id", "name", "zone", "lat", "lon", "free_flow_speed"]],
                             on="junction_id")

junctions.to_csv("junctions_cameras.csv", index=False)
traffic_full.to_csv("traffic_flow.csv", index=False)
v.to_csv("violations.csv", index=False)
challans.to_csv("challans.csv", index=False)
enriched.to_csv("violations_enriched.csv", index=False)

print("traffic rows:", len(traffic_full), "| violations:", len(v), "| challans:", len(challans))
print(challans.status.value_counts())

from google.colab import files
for f in ["junctions_cameras.csv", "traffic_flow.csv", "violations.csv",
          "challans.csv", "violations_enriched.csv"]:
    files.download(f)

traffic rows: 86880 | violations: 32000 | challans: 32000
status
Paid         23007
Overdue       6323
Pending       2128
Contested      542
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import os
if not os.path.exists("violations.csv"):
    from google.colab import files
    print("Please select violations.csv and traffic_flow.csv")
    files.upload()

violations = pd.read_csv("violations.csv", parse_dates=["timestamp"])
traffic = pd.read_csv("traffic_flow.csv", parse_dates=["timestamp"])

END = pd.Timestamp("2026-10-28")

def daily_series(dates, values=None):
    df = pd.DataFrame({"date": dates.dt.normalize(), "val": 1 if values is None else values})
    s = df.groupby("date")["val"].sum()
    s = s[s.index <= END]
    return s.reindex(pd.date_range(s.index.min(), END, freq="D"), fill_value=0)

series = {
    "Daily Violations": (daily_series(violations.timestamp), 30),
    "Daily Vehicle Volume": (daily_series(traffic.timestamp, traffic.vehicle_count), 14),
}

def mape(actual, pred):
    return float((np.abs(actual - pred) / actual).mean() * 100)

def mae(actual, pred):
    return float(np.abs(actual - pred).mean())

def hw_model(y):
    return ExponentialSmoothing(y, trend="add", damped_trend=True,
                                seasonal="add", seasonal_periods=7).fit()

all_rows, acc_rows = [], []

for name, (y, horizon) in series.items():
    # ----- Test accuracy: hold out the last 30 days -----
    train, test = y.iloc[:-30], y.iloc[-30:]

    # Baseline: seasonal naive (same weekday, last week)
    naive = pd.Series(np.tile(train.iloc[-7:].values, 5)[:30], index=test.index)
    # Holt-Winters
    hw_pred = hw_model(train).forecast(30)

    acc_rows.append({"metric": name, "method": "Seasonal Naive (baseline)",
                     "MAPE_%": round(mape(test, naive), 2), "MAE": round(mae(test, naive), 2)})
    acc_rows.append({"metric": name, "method": "Holt-Winters",
                     "MAPE_%": round(mape(test, hw_pred), 2), "MAE": round(mae(test, hw_pred), 2)})

    final = hw_model(y)
    fc = final.forecast(horizon)
    resid_sd = (y - final.fittedvalues).std()
    lower = (fc - 1.96 * resid_sd).clip(lower=0)
    upper = fc + 1.96 * resid_sd

    hist = pd.DataFrame({"date": y.index, "metric": name, "actual": y.values,
                         "forecast": np.nan, "lower": np.nan, "upper": np.nan})
    fut = pd.DataFrame({"date": fc.index, "metric": name, "actual": np.nan,
                        "forecast": fc.round(0).values, "lower": lower.round(0).values,
                        "upper": upper.round(0).values})
    all_rows += [hist, fut]

forecast = pd.concat(all_rows, ignore_index=True)
accuracy = pd.DataFrame(acc_rows)

forecast.to_csv("forecast.csv", index=False)
accuracy.to_csv("forecast_accuracy.csv", index=False)

print(accuracy.to_string(index=False))
print()
print(forecast[forecast.forecast.notna()].groupby("metric")[["forecast"]].agg(["min", "max", "mean"]).round(0))

from google.colab import files
files.download("forecast.csv")
files.download("forecast_accuracy.csv")

              metric                    method  MAPE_%      MAE
    Daily Violations Seasonal Naive (baseline)   11.31    11.60
    Daily Violations              Holt-Winters   11.30    11.89
Daily Vehicle Volume Seasonal Naive (baseline)    9.32 11040.13
Daily Vehicle Volume              Holt-Winters   10.30 12182.93

                      forecast                    
                           min       max      mean
metric                                            
Daily Vehicle Volume  101531.0  126719.0  119122.0
Daily Violations          87.0     108.0     102.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>